In [1]:
import os
import sys

import hydra
import numpy as np
import pandas as pd
import wandb
from omegaconf import DictConfig, OmegaConf
from sklearn.model_selection import GroupKFold
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.dataset import list_well_ids, load_well, load_typewell
from src.features import build_feature_frame
from src.model import LGBMResidualModel
NON_FEATURE_COLS = ["well_id", "row_index", "is_tail", "TVT", "resid", "tvt_diff_abs"]

In [2]:
def build_all_features(train_dir: str, gr_rolling_windows: list[int]) -> pd.DataFrame:
    well_ids = list_well_ids(train_dir)
    frames = []
    for well_id in well_ids:
        hw = load_well(well_id, train_dir)
        tw = load_typewell(well_id, train_dir)
        feat = build_feature_frame(hw, tw, gr_rolling_windows)
        feat["tvt_diff_abs"] = hw["TVT"].diff().abs().to_numpy()
        frames.append(feat)
    return pd.concat(frames, ignore_index=True)

In [3]:

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [7]:
sys.argv = ['']

In [8]:
@hydra.main(version_base=None, config_path="/workspace/conf", config_name="config")
def main(cfg: DictConfig) -> None:
    print(OmegaConf.to_yaml(cfg))


    df = build_all_features(cfg.data.train_dir, cfg.train.gr_rolling_windows)

    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

    tail_df = df[df["is_tail"]].reset_index(drop=True)

    return df 

In [9]:
df = main()

model:
  name: lgbm
  params:
    objective: regression
    metric: rmse
    learning_rate: 0.03
    num_leaves: 31
    max_depth: -1
    min_data_in_leaf: 50
    feature_fraction: 0.8
    bagging_fraction: 0.8
    bagging_freq: 1
    lambda_l2: 1.0
    verbosity: -1
    seed: 42
  num_boost_round: 3000
  early_stopping_rounds: 100
data:
  train_dir: /workspace/data/raw/train
  test_dir: /workspace/data/raw/test
  n_folds: 5
  seed: 42
  tvt_min: 9245.0
  tvt_max: 12894.0
train:
  rare_threshold: 0.91
  train_on_tail_only: true
  gr_rolling_windows:
  - 10
  - 30
  - 100
wandb:
  project: baseline
  run_name: ???



In [14]:
df = pd.DataFrame(df)

In [15]:
df.head(2)

""
